# BrainIAC embeddings for the Parkinson's cross-cohort study (full run)
Settings before **Save & Run All (Commit)**: Accelerator **GPU T4**, Internet **On**.
Inputs to attach: the dataset with `BrainIAC.ckpt` (eval_data), the dataset with `volumes.csv` and `subject_table_with_demographics.csv`, and the dataset with `pd_embedding_eval.py`.
Cells run in order: 1 install, 2 scans, 3 registration, 4 skull-stripping, 5 embeddings, 6 checks, 7 evaluation.

In [ ]:
# 1. Install, clone BrainIAC, add compatibility shims
!pip -q install boto3 nibabel SimpleITK einops scikit-image tqdm antspyx
!pip -q install monai==1.3.2 --no-deps
!test -d /tmp/BrainIAC || git clone -q --depth 1 https://github.com/AIM-KannLab/BrainIAC.git /tmp/BrainIAC

import os
os.makedirs("/tmp/shim", exist_ok=True)
open("/tmp/shim/imp.py", "w").write("""import importlib.util, sys
def load_source(name, pathname):
    spec = importlib.util.spec_from_file_location(name, pathname)
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod
    spec.loader.exec_module(mod); return mod
""")
open("/tmp/shim/sitecustomize.py", "w").write("""import torch
_orig = torch.load
def _load(*a, **k):
    k.setdefault("weights_only", False)
    return _orig(*a, **k)
torch.load = _load
""")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
from monai.networks.nets import ViT
print("monai ViT import OK")
assert os.path.exists("/tmp/BrainIAC/src/preprocessing/atlases/temp_head.nii.gz"), "BrainIAC clone incomplete"

In [ ]:
# 2. Rebuild the 83 T1 scans (set LIMIT = 2 only for a quick test)
import glob, re, tarfile, shutil, boto3, pandas as pd
from botocore import UNSIGNED
from botocore.config import Config
LIMIT = None
RAW, UNPROC, OUT = "/tmp/pd_raw", "/tmp/bi/unprocessed", "/kaggle/working/brainiac"
for d in (RAW, UNPROC, OUT): os.makedirs(d, exist_ok=True)
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
rows = []
for cohort in ["neurocon", "taowu"]:
    arc = f"/tmp/{cohort}.tar.gz"
    if not os.path.exists(arc):
        s3.download_file("fcp-indi", f"data/Projects/INDI/umf_pd/{cohort}.tar.gz", arc)
    dest = f"{RAW}/{cohort}"
    if not glob.glob(f"{dest}/**/anat/*T1w.nii.gz", recursive=True):
        os.makedirs(dest, exist_ok=True)
        with tarfile.open(arc, "r:gz") as tar:
            members = [m for m in tar.getmembers() if m.isfile() and "t1w.nii" in m.name.lower()]
            tar.extractall(path=dest, members=members, filter="data")
    paths = sorted(glob.glob(f"{dest}/**/anat/*T1w.nii.gz", recursive=True))
    for path in (paths[:LIMIT] if LIMIT else paths):
        sid = re.search(r"(sub-[^/_]+)", path).group(1)
        grp = re.search(r"sub-(control|patient)", sid).group(1)
        shutil.copy(path, f"{UNPROC}/{cohort}_{sid}.nii.gz")
        rows.append(dict(cohort=cohort, sid=sid, label=0 if grp == "control" else 1))
meta = pd.DataFrame(rows)
print(meta.groupby(["cohort", "label"]).size())
if not LIMIT:
    assert len(meta) == 83, f"expected 83 scans, found {len(meta)}"

In [ ]:
# 3. N4 bias correction + rigid registration to BrainIAC's template (ANTs; the slow step)
import ants, time
from tqdm import tqdm
TEMP = "/tmp/BrainIAC/src/preprocessing/atlases/temp_head.nii.gz"
REG = "/tmp/bi/registered"
os.makedirs(REG, exist_ok=True)
fixed = ants.image_read(TEMP)
failed = []
for k, f in enumerate(tqdm(sorted(glob.glob("/tmp/bi/unprocessed/*.nii.gz")))):
    ID = os.path.basename(f).split(".")[0]
    out = f"{REG}/{ID}_0000.nii.gz"
    if os.path.exists(out):
        continue
    t0 = time.time()
    try:
        mov = ants.n4_bias_field_correction(ants.image_read(f))
        reg = ants.registration(fixed=fixed, moving=mov, type_of_transform="Rigid", random_seed=42)
        ants.image_write(reg["warpedmovout"], out)
    except Exception as e:
        failed.append((ID, str(e)[:200])); print("FAILED", ID, e)
    if k < 3:
        print(f"{ID}: {time.time() - t0:.0f} s", flush=True)
n_reg = len(glob.glob(f"{REG}/*_0000.nii.gz"))
print(n_reg, "registered (want", len(meta), ")")
assert not failed and n_reg == len(meta), f"registration incomplete: {failed[:3]}"

In [ ]:
# 4. Skull-stripping with HD-BET (one-line numpy patch, then run)
import subprocess
p = "/tmp/BrainIAC/src/preprocessing/HD_BET/data_loading.py"
s = open(p).read()
old = "np.any(np.array(seg_old_size) != np.array(dct['size'])[[2, 1, 0]])"
new = "np.any(np.array(seg_old_size.shape) != np.array(dct['size'])[[2, 1, 0]])"
if old in s:
    open(p, "w").write(s.replace(old, new))
assert new in open(p).read(), "HD-BET patch missing"
PROC = "/kaggle/working/brainiac/processed"
os.makedirs(PROC, exist_ok=True)
dev = "0" if torch.cuda.is_available() else "cpu"
cmd = ("cd /tmp/BrainIAC/src/preprocessing && PYTHONPATH=/tmp/shim python -u -c "
       f"\"from HD_BET.hd_bet import hd_bet; hd_bet('{REG}', '{PROC}', device='{dev}', mode='fast', tta=0)\"")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(r.stdout[-600:]); print(r.stderr[-1200:])
assert r.returncode == 0, "HD-BET failed"
n_proc = len(glob.glob(f"{PROC}/*_0000.nii.gz"))
print(n_proc, "skull-stripped scans (want", len(meta), ")")
assert n_proc == len(meta), "skull-stripping incomplete"

In [ ]:
# 5. BrainIAC embeddings (frozen, no fine-tuning)
ck = glob.glob("/kaggle/input/**/BrainIAC.ckpt", recursive=True)
assert ck, "add the dataset containing BrainIAC.ckpt as an input"
meta["pat_id"] = meta.cohort + "_" + meta.sid + "_0000"
meta[["pat_id", "label"]].to_csv("/tmp/bi/pat.csv", index=False)
!cd /tmp/BrainIAC/src && PYTHONPATH=/tmp/shim python -u get_brainiac_features.py --checkpoint {ck[0]} --input_csv /tmp/bi/pat.csv --output_csv /tmp/bi/features_raw.csv --root_dir /kaggle/working/brainiac/processed
feats = pd.read_csv("/tmp/bi/features_raw.csv")
assert len(feats) == len(meta) and (feats.GroundTruthClassLabel.values == meta.label.values).all()
emb = pd.concat([meta[["cohort", "sid", "label"]].reset_index(drop=True),
                 feats.drop(columns=["GroundTruthClassLabel"])], axis=1)
emb.to_csv("/kaggle/working/brainiac/embeddings_brainiac.csv", index=False)
print(emb.shape, "(want", len(meta), "x 771)")

In [ ]:
# 6. Sanity checks on the embeddings
import numpy as np
F = emb.filter(like="Feature_")
assert np.isfinite(F.values).all(), "non-finite values in the embeddings"
print("feature std across scans:", round(float(F.std().mean()), 4), "(must be clearly above 0)")
assert float(F.std().mean()) > 1e-3, "embeddings look degenerate"
d = np.sqrt(((F.values[:, None] - F.values[None]) ** 2).sum(-1))
print("median distance between different scans:", round(float(np.median(d[np.triu_indices(len(F), 1)])), 2))

In [ ]:
# 7. Evaluation under the pre-declared protocol (does not fail the run if an input is missing)
vol  = glob.glob("/kaggle/input/**/volumes.csv", recursive=True)
demo = glob.glob("/kaggle/input/**/subject_table_with_demographics.csv", recursive=True)
code = glob.glob("/kaggle/input/**/pd_embedding_eval.py", recursive=True)
print("volumes:", vol, "| demographics:", demo, "| script:", code)
if vol and demo and code:
    cmd = (f"python -u {code[0]} --vol {vol[0]} --demo {demo[0]} "
           "--emb brainiac=/kaggle/working/brainiac/embeddings_brainiac.csv --out /kaggle/working/brainiac")
    get_ipython().system(cmd)
else:
    print("Evaluation skipped: attach the datasets holding volumes.csv, the demographics table and "
          "pd_embedding_eval.py, then rerun only this cell. The embeddings are already saved in /kaggle/working/brainiac.")
print(sorted(os.listdir("/kaggle/working/brainiac")))